# MMD-aligned LSTM — Contribution 3 (drop-in for `modeling_v6_final.ipynb`)

This notebook adds a third domain-adaptation method (Maximum Mean Discrepancy alignment) to compare against your existing DANN and GRL approaches.

## Prerequisites — must be true when you run this notebook

1. **`modeling_v6_final.ipynb` has already been run end-to-end** in the same Jupyter kernel — or — these cells have been pasted into the v6 notebook itself.
2. The following variables exist in your kernel namespace:
   - `nn`, `optim`, `torch`, `np`, `pd`, `plt`, `Path`, `Optional`, `copy`, `time`, `math`
   - `tqdm`, `clear_output`, `display`, `DataLoader`
   - `Device`, `USE_SENSORS`, `W`, `DANN_BATCH`, `SEEDS`, `CONFIGS`, `CKPT_DIR`
   - `ImprovedLSTM`, `CMAPSSWindowDataset`, `StageAwareBatchSampler`
   - `asymmetric_mse`, `evaluate_loader`, `run_inference`, `score_predictions`
   - `collate_from_dataset`, `ganin_lambda`, `set_seed`, `make_splits`
   - `splits`, `fd1_train`, `fd2_train_global`, `fd2_train_cond`, `fd1_test`, `fd2_test_global`, `fd2_test_cond`, `fd1_true_rul`, `fd2_true_rul`
   - `lower_bound_metrics`, `ub_fd2_metrics`, `dann_fd2_metrics`, `grl_fd2_metrics` (for gap comparison)

If you opened a fresh kernel and ran *only* this notebook, **it will fail** with `NameError`. That is by design — we share v6's heavy state instead of duplicating ~600 lines of setup.

## Why MMD?

Per the 2025 DA survey for turbofan RUL (arxiv 2510.03604): metric-based methods like MMD offer "lower computational complexity and reduced risk of mode collapse, making them more stable and efficient compared to adversarial approaches" (i.e., GRL/DANN).

Your GRL and DANN both closed ~14% of the FD001→FD002 gap. MMD typically achieves comparable or slightly better gap closure with smoother training. This gives you a **3-method comparison** in your report instead of two adversarial variants doing the same thing.

## Sanity-check imports (run me first)

The cell below verifies your prerequisites are loaded. If it fails, go back and run `modeling_v6_final.ipynb` first.


In [6]:
import os, time, math, copy
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import DataLoader

from tqdm.auto import tqdm
from IPython.display import display, clear_output

In [7]:
# ── Sanity check: confirm prerequisites are in the kernel ─────────────────
_required = [
    "nn", "optim", "torch", "np", "pd", "plt", "Path", "Optional", "copy",
    "tqdm", "clear_output", "display", "DataLoader",
    "Device", "USE_SENSORS", "W", "DANN_BATCH", "SEEDS", "CONFIGS", "CKPT_DIR",
    "ImprovedLSTM", "CMAPSSWindowDataset", "StageAwareBatchSampler",
    "asymmetric_mse", "evaluate_loader", "run_inference", "score_predictions",
    "collate_from_dataset", "ganin_lambda", "set_seed", "make_splits",
    "splits", "lower_bound_metrics", "ub_fd2_metrics",
]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise NameError(
        f"Missing {len(_missing)} prerequisite(s): {_missing}\n"
        "Run modeling_v6_final.ipynb to completion first (in the same kernel)."
    )
print(f"All {len(_required)} prerequisites present. Proceeding with MMD.")


NameError: Missing 34 prerequisite(s): ['nn', 'optim', 'torch', 'np', 'pd', 'plt', 'Path', 'Optional', 'copy', 'tqdm', 'clear_output', 'display', 'DataLoader', 'Device', 'USE_SENSORS', 'W', 'DANN_BATCH', 'SEEDS', 'CONFIGS', 'CKPT_DIR', 'ImprovedLSTM', 'CMAPSSWindowDataset', 'StageAwareBatchSampler', 'asymmetric_mse', 'evaluate_loader', 'run_inference', 'score_predictions', 'collate_from_dataset', 'ganin_lambda', 'set_seed', 'make_splits', 'splits', 'lower_bound_metrics', 'ub_fd2_metrics']
Run modeling_v6_final.ipynb to completion first (in the same kernel).

## Cell A · MMD loss + `LSTMWithMMD` model

Multi-kernel Gaussian MMD is the canonical metric for deep domain adaptation (Long et al. 2015, "Learning Transferable Features with Deep Adaptation Networks"). Conceptually:

$$\text{MMD}^2(P_{src}, P_{tgt}) = \mathbb{E}_{s,s'}[k(s,s')] + \mathbb{E}_{t,t'}[k(t,t')] - 2 \mathbb{E}_{s,t}[k(s,t)]$$

where $k$ is a sum of Gaussian kernels at multiple bandwidths. Smaller MMD = more aligned distributions in the RKHS sense. We minimize MMD between encoded source and target features alongside the supervised RUL loss.

The model has **no domain classifier** — alignment comes directly from the kernel-distance loss. No min-max game, no GRL.


In [ ]:
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    Multi-kernel Gaussian kernel matrix between source and target.
    Standard DA recipe (Long et al. 2015, Tzeng et al. 2014).

    Args:
        source: (n_src, d) tensor
        target: (n_tgt, d) tensor
        kernel_num: number of kernels in the linear combination
        kernel_mul: multiplicative spacing between bandwidths

    Returns:
        (n_src+n_tgt, n_src+n_tgt) summed-kernel matrix
    """
    n_samples = int(source.size(0)) + int(target.size(0))
    total = torch.cat([source, target], dim=0)

    # Pairwise squared L2 distances
    total0 = total.unsqueeze(0).expand(n_samples, -1, -1)
    total1 = total.unsqueeze(1).expand(-1, n_samples, -1)
    L2_distance = ((total0 - total1) ** 2).sum(dim=2)

    # Bandwidth: median heuristic if fix_sigma not provided
    if fix_sigma is None:
        bandwidth = torch.sum(L2_distance.detach()) / (n_samples ** 2 - n_samples)
    else:
        bandwidth = fix_sigma

    # Spread bandwidths around the median to span small-to-large scales
    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    # Sum of Gaussian kernels with different bandwidths
    kernel_val = sum(torch.exp(-L2_distance / b) for b in bandwidth_list)
    return kernel_val


def mmd_loss(source_features, target_features, kernel_mul=2.0, kernel_num=5):
    """
    Multi-kernel Maximum Mean Discrepancy between source and target embeddings.
    Smaller MMD = more aligned distributions.

    The empirical MMD^2 (Gretton et al. 2012) decomposes as:
        MMD^2 = E[k(s,s')] + E[k(t,t')] - 2 E[k(s,t)]
    """
    n_src = source_features.size(0)
    n_tgt = target_features.size(0)
    kernels = gaussian_kernel(source_features, target_features, kernel_mul, kernel_num)

    XX = kernels[:n_src, :n_src]
    YY = kernels[n_src:, n_src:]
    XY = kernels[:n_src, n_src:]
    YX = kernels[n_src:, :n_src]

    loss = XX.mean() + YY.mean() - XY.mean() - YX.mean()
    return loss


class LSTMWithMMD(nn.Module):
    """
    BiLSTM (same encoder as ImprovedLSTM) + MMD-based feature alignment.
    No domain classifier — alignment is enforced by minimizing MMD between
    encoded source and encoded target features directly.

    Simpler than DANN/GRL: no min-max game, smoother convergence, no GRL.
    """
    def __init__(self, n_features, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm_body = ImprovedLSTM(n_features, hidden, num_layers, dropout,
                                      bidirectional=True)

    def encode(self, x):
        """Return encoder features (B, 256)."""
        return self.lstm_body.encode(x)

    def predict_rul(self, x):
        """RUL prediction (used at inference)."""
        return self.lstm_body(x)

    def forward(self, x_src, x_tgt):
        """Returns (rul_pred_src, h_src, h_tgt) for joint MSE+MMD training."""
        h_src = self.lstm_body.encode(x_src)
        h_tgt = self.lstm_body.encode(x_tgt)
        rul_pred = self.lstm_body.head(h_src).squeeze(-1)
        return rul_pred, h_src, h_tgt


# Sanity-check the model and the MMD loss
_m = LSTMWithMMD(len(USE_SENSORS)).to(Device)
print(f"LSTM-MMD params: {sum(p.numel() for p in _m.parameters()):,}")

_a = torch.randn(64, 256, device=Device)
_b = torch.randn(64, 256, device=Device) + 0.5
print(f"MMD self-distance (should be ~0)  : {mmd_loss(_a, _a).item():.6f}")
print(f"MMD shifted distance (should be >0): {mmd_loss(_a, _b).item():.6f}")
del _m, _a, _b


NameError: name 'USE_SENSORS' is not defined

## Cell B · `MMDTracker` and `train_mmd_two_phase`

Same two-phase schedule as your `train_grl_two_phase`:

- **Phase 1** (epochs 1..15): λ=0, pure asymmetric-MSE on source. RUL head and encoder converge to the supervised solution first.
- **Phase 2** (epochs 16..50): λ ramps via the Ganin schedule. MMD pressure pulls source/target encodings together; the RUL head prevents collapse to trivial constant features.

Total loss: `asymmetric_mse(rul_pred, y_src) + lambda * MMD(h_src, h_tgt)`


In [ ]:
class MMDTracker:
    """4-panel live tracker: total / val / RUL / MMD. Mirrors DANNTracker."""
    def __init__(self, n_epochs, plot_freq=1):
        self.n_epochs = n_epochs; self.plot_freq = plot_freq; self.epoch = 0
        self.total_loss, self.rul_loss, self.mmd_loss_v, self.val_rmse, self.lam_hist = [],[],[],[],[]
        self._stopped_at = None; self.t0 = time.time()
        plt.ioff()
        self.fig, axes = plt.subplots(2, 2, figsize=(13, 7))
        (self.ax_tot, self.ax_val), (self.ax_rul, self.ax_mmd) = axes
        self.c_tot, = self.ax_tot.plot([], [], color="steelblue")
        self.c_val, = self.ax_val.plot([], [], color="darkorange")
        self.c_rul, = self.ax_rul.plot([], [], color="seagreen")
        self.c_mmd, = self.ax_mmd.plot([], [], color="mediumpurple")
        self.ax_mmd2 = self.ax_mmd.twinx()
        self.c_lam,  = self.ax_mmd2.plot([], [], color="gray", linestyle="--", alpha=0.7)
        for ax, t, yl in [(self.ax_tot,"Total loss","Loss"),
                          (self.ax_val,"Val RMSE (FD001)","RMSE"),
                          (self.ax_rul,"RUL loss (asym MSE)","MSE"),
                          (self.ax_mmd,"MMD loss + lambda","MMD")]:
            ax.set_title(t); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
            ax.set_xlim(0, n_epochs+1); ax.grid(linestyle="--", alpha=0.6)
        self.ax_mmd2.set_ylabel("lambda"); plt.tight_layout()

    def update_epoch(self, tot, rul, mmd_v, vr, lam):
        self.total_loss.append(tot); self.rul_loss.append(rul)
        self.mmd_loss_v.append(mmd_v); self.val_rmse.append(vr)
        self.lam_hist.append(lam); self.epoch += 1
        if self.epoch % self.plot_freq == 0 or self.epoch == self.n_epochs:
            self._redraw()

    def mark_early_stop(self, ep): self._stopped_at = ep

    def _redraw(self):
        xs = list(range(1, self.epoch+1))
        for c, d, ax in [(self.c_tot,self.total_loss,self.ax_tot),
                         (self.c_val,self.val_rmse, self.ax_val),
                         (self.c_rul,self.rul_loss, self.ax_rul),
                         (self.c_mmd,self.mmd_loss_v,self.ax_mmd)]:
            c.set_data(xs, d); ax.relim(); ax.autoscale_view()
        self.c_lam.set_data(xs, self.lam_hist)
        self.ax_mmd2.relim(); self.ax_mmd2.autoscale_view()
        if self._stopped_at:
            for ax in (self.ax_tot, self.ax_val, self.ax_rul, self.ax_mmd):
                ax.axvline(self._stopped_at, color="crimson", linestyle=":", lw=1.5)
        elapsed = time.time()-self.t0
        eta = (elapsed/self.epoch)*(self.n_epochs-self.epoch)
        self.fig.suptitle(
            f"MMD  Ep {self.epoch}/{self.n_epochs}  val RMSE={self.val_rmse[-1]:.2f}  "
            f"lam={self.lam_hist[-1]:.3f}  elapsed {elapsed:.0f}s  ETA {eta:.0f}s"
            + ("  [early stop]" if self._stopped_at else ""), fontsize=10)
        clear_output(wait=True); display(self.fig)

    def close(self): plt.close(self.fig)


def train_mmd_two_phase(
    model,
    src_dataset,
    tgt_dataset,
    val_loader,
    phase1_epochs=15,
    phase2_epochs=35,
    lr=5e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    patience=20,
    lambda_max=1.0,
    late_weight=1.5,
    batch_size=128,
    kernel_num=5,
    kernel_mul=2.0,
    device=None,
    ckpt_path=None,
    seed=42,
    silent=False,
):
    """
    Two-phase MMD training mirroring train_grl_two_phase.

    Phase 1 (epochs 1..phase1_epochs): lambda=0, pure asymmetric-MSE on source.
    Phase 2 (epochs phase1+1..end): lambda ramps via Ganin schedule.

    Loss = asym_MSE(rul_pred, y_src) + lambda * MMD(h_src, h_tgt)
    """
    if device is None: device = Device
    n_epochs = phase1_epochs + phase2_epochs
    model    = model.to(device)
    opt      = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    sampler  = StageAwareBatchSampler(src_dataset, tgt_dataset,
                                      batch_size=batch_size, rng_seed=seed)
    tracker  = None if silent else MMDTracker(n_epochs)
    history  = []
    best_val, best_ep, no_imp = float("inf"), 0, 0
    best_state = None

    pbar = tqdm(range(1, n_epochs+1), desc="MMD-2phase", unit="epoch", disable=silent)
    for epoch in pbar:
        # Lambda schedule
        if epoch <= phase1_epochs:
            lam = 0.0
        else:
            p2_epoch = epoch - phase1_epochs
            lam = ganin_lambda(p2_epoch, phase2_epochs, high=lambda_max)

        model.train()
        ep_tot, ep_rul, ep_mmd, ep_n = 0.0, 0.0, 0.0, 0

        for src_idx, tgt_idx in sampler:
            x_src, y_src = collate_from_dataset(src_dataset, src_idx)
            x_tgt, _     = collate_from_dataset(tgt_dataset, tgt_idx)
            x_src, y_src = x_src.to(device), y_src.to(device)
            x_tgt        = x_tgt.to(device)

            rul_pred, h_src, h_tgt = model(x_src, x_tgt)
            B_src = x_src.size(0)

            l_rul = asymmetric_mse(rul_pred, y_src, late_weight)
            if lam > 0:
                l_mmd = mmd_loss(h_src, h_tgt, kernel_mul=kernel_mul, kernel_num=kernel_num)
            else:
                # Phase 1: log MMD without grad path
                with torch.no_grad():
                    l_mmd = mmd_loss(h_src, h_tgt, kernel_mul=kernel_mul, kernel_num=kernel_num)
            loss = l_rul + lam * l_mmd

            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            ep_tot += loss.item()*B_src
            ep_rul += l_rul.item()*B_src
            ep_mmd += float(l_mmd.item())*B_src
            ep_n  += B_src

        sched.step()
        _, val_rmse = evaluate_loader(model, val_loader, device, mode="dann")

        phase_str = "P1" if epoch <= phase1_epochs else "P2"
        history.append(dict(epoch=epoch, phase=phase_str,
                            total_loss=ep_tot/ep_n, rul_loss=ep_rul/ep_n,
                            mmd_loss=ep_mmd/ep_n, val_rmse=val_rmse, lam=lam))
        if tracker: tracker.update_epoch(ep_tot/ep_n, ep_rul/ep_n, ep_mmd/ep_n, val_rmse, lam)
        if not silent:
            pbar.set_postfix({"phase": phase_str, "val": f"{val_rmse:.2f}",
                              "lam": f"{lam:.3f}", "no_imp": no_imp})

        if val_rmse < best_val:
            best_val, best_ep, no_imp = val_rmse, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
            if ckpt_path:
                torch.save({"model": best_state, "epoch": epoch,
                            "val_rmse": val_rmse}, ckpt_path)
        else:
            no_imp += 1
            if no_imp >= patience:
                if not silent: print(f"\nEarly stopping at epoch {epoch}.")
                if tracker: tracker.mark_early_stop(epoch); tracker._redraw()
                break

    if tracker: tracker.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"Best val RMSE = {best_val:.3f} at epoch {best_ep}")
    return model, pd.DataFrame(history)


## Cell C · 3-seed sweep

Runs identically to your DANN/GRL sweeps: 3 seeds × CONFIGS combinations. Each run takes ~5-7 min on M4-MPS.

**Total expected wall time: 50-70 minutes.** Don't sit and watch — start the report Methods section while this runs.


In [ ]:
mmd_sweep_results = []

for seed in SEEDS:
    set_seed(seed)
    sp = make_splits(fd1_train, fd2_train_global, fd2_train_cond,
                     fd1_test, fd2_test_global, fd2_test_cond,
                     fd1_true_rul, fd2_true_rul, seed=seed)
    src_val_loader = DataLoader(sp["fd1_val_set"], batch_size=256, shuffle=False, num_workers=0)

    for cfg in CONFIGS:
        set_seed(seed)
        tag = f"mmd_seed{seed}_drop{cfg['dropout']}_clip{cfg['grad_clip']}"
        print(f"\n── MMD sweep: {tag} ──")

        mmd_model = LSTMWithMMD(n_features=len(USE_SENSORS), dropout=cfg["dropout"])
        mmd_model, hist = train_mmd_two_phase(
            mmd_model,
            src_dataset=sp["fd1_train_set"],
            tgt_dataset=sp["tgt_train_set"],
            val_loader=src_val_loader,
            phase1_epochs=15, phase2_epochs=35,
            lr=5e-4, weight_decay=1e-4,
            grad_clip=cfg["grad_clip"], patience=20,
            lambda_max=1.0, late_weight=1.5,
            batch_size=DANN_BATCH,
            device=Device,
            ckpt_path=CKPT_DIR / f"{tag}.pt",
            seed=seed,
        )

        raw_fd2 = run_inference(mmd_model, sp["X_test_fd2_global"], Device, mode="dann")
        raw_fd1 = run_inference(mmd_model, sp["X_test_fd1"],        Device, mode="dann")
        m_fd2   = score_predictions(sp["y_test_fd2"], raw_fd2, f"{tag} FD002")
        m_fd1   = score_predictions(sp["y_test_fd1"], raw_fd1, f"{tag} FD001")
        mmd_sweep_results.append(dict(tag=tag, seed=seed, **cfg,
                                      fd2_rmse=m_fd2["rmse"], fd2_score=m_fd2["score"],
                                      fd1_rmse=m_fd1["rmse"],
                                      best_val=hist["val_rmse"].min()))

mmd_sweep_df = pd.DataFrame(mmd_sweep_results).sort_values("fd2_rmse")
print("\n── MMD sweep results (sorted by FD002 RMSE) ──")
print(mmd_sweep_df[["tag","best_val","fd1_rmse","fd2_rmse","fd2_score"]].to_string(index=False))


## Cell D · Best MMD evaluation + gap-closure comparison

Loads the winning MMD checkpoint, evaluates on FD002 and FD001 test sets, and computes the gap-closure percentages against your `lower_bound_metrics` (FD001-LSTM applied naively to FD002) and `ub_fd2_metrics` (LSTM trained directly on FD002).

Also persists `mmd_sweep.csv` to `CKPT_DIR/` for the report.


In [ ]:
best_mmd_row  = mmd_sweep_df.iloc[0]
best_mmd_tag  = best_mmd_row["tag"]
best_mmd_ckpt = CKPT_DIR / f"{best_mmd_tag}.pt"

mmd_best = LSTMWithMMD(n_features=len(USE_SENSORS), dropout=best_mmd_row["dropout"])
sd = torch.load(best_mmd_ckpt, map_location=Device, weights_only=False)
mmd_best.load_state_dict(sd["model"])
mmd_best = mmd_best.to(Device)

raw_mmd_fd2 = run_inference(mmd_best, splits["X_test_fd2_global"], Device, mode="dann")
raw_mmd_fd1 = run_inference(mmd_best, splits["X_test_fd1"],        Device, mode="dann")
mmd_fd2_metrics = score_predictions(splits["y_test_fd2"], raw_mmd_fd2,
                                    f"Best MMD ({best_mmd_tag}) on FD002")
mmd_fd1_metrics = score_predictions(splits["y_test_fd1"], raw_mmd_fd1,
                                    f"Best MMD ({best_mmd_tag}) on FD001")

# Compute gap closure
lb_rmse = lower_bound_metrics["rmse"]
ub_rmse = ub_fd2_metrics["rmse"]
lb_score = lower_bound_metrics["score"]
ub_score = ub_fd2_metrics["score"]

mmd_gap_rmse = (lb_rmse - mmd_fd2_metrics["rmse"]) / (lb_rmse - ub_rmse) * 100
mmd_gap_score = (lb_score - mmd_fd2_metrics["score"]) / (lb_score - ub_score) * 100

print(f"\n══════════════════════════════════════════════════════════")
print(f"  Method                FD002 RMSE   FD002 Score   Gap (RMSE)")
print(f"══════════════════════════════════════════════════════════")
print(f"  Lower bound (no DA)   {lb_rmse:>10.2f}   {lb_score:>11.0f}        0.0%")
try:
    dann_rmse_str = f"{dann_fd2_metrics['rmse']:>10.2f}"
    dann_score_str = f"{dann_fd2_metrics['score']:>11.0f}"
    dann_gap = (lb_rmse - dann_fd2_metrics['rmse']) / (lb_rmse - ub_rmse) * 100
    print(f"  DANN (stage-aware)    {dann_rmse_str}   {dann_score_str}    {dann_gap:>5.1f}%")
except NameError:
    print(f"  DANN (stage-aware)    [dann_fd2_metrics not in namespace]")
try:
    grl_rmse_str = f"{grl_fd2_metrics['rmse']:>10.2f}"
    grl_score_str = f"{grl_fd2_metrics['score']:>11.0f}"
    grl_gap = (lb_rmse - grl_fd2_metrics['rmse']) / (lb_rmse - ub_rmse) * 100
    print(f"  LSTM-GRL (2-phase)    {grl_rmse_str}   {grl_score_str}    {grl_gap:>5.1f}%")
except NameError:
    print(f"  LSTM-GRL (2-phase)    [grl_fd2_metrics not in namespace]")
print(f"  LSTM-MMD (2-phase)    {mmd_fd2_metrics['rmse']:>10.2f}   {mmd_fd2_metrics['score']:>11.0f}    {mmd_gap_rmse:>5.1f}%")
print(f"  Upper bound (super.)  {ub_rmse:>10.2f}   {ub_score:>11.0f}      100.0%")
print(f"══════════════════════════════════════════════════════════")

mmd_sweep_df.to_csv(CKPT_DIR / "mmd_sweep.csv", index=False)
print(f"\nSaved → {CKPT_DIR / 'mmd_sweep.csv'}")
